# TinyMoreh — GPU Training

Trains the TinyMoreh transformer on a free Colab GPU.

**Before running:** Runtime → Change runtime type → select **T4 GPU**.

In [ ]:
# Clone repo and download data
!git clone https://github.com/Robespierre17/tiny-moreh.git
%cd /content/tiny-moreh
!python data/download_data.py

In [ ]:
# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Go to Runtime -> Change runtime type -> GPU")

In [ ]:
import sys
sys.path.insert(0, '/content/tiny-moreh/src')

import os
import re
import time
import torch

# Load raw corpus
with open('data/maimonides_corpus.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()
print(f"Raw corpus: {len(raw_text):,} characters, {len(set(raw_text))} unique")

# Filter to printable ASCII only — strips Hebrew/Greek fragments that
# inflate vocab size and confuse a small model
allowed = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,;:!\'\"()-\n\t')
text = ''.join(ch if ch in allowed else '' for ch in raw_text)
text = re.sub(r' +', ' ', text)
text = re.sub(r'\n{3,}', '\n\n', text)
text = text.strip()
print(f"Clean corpus: {len(text):,} characters, {len(set(text))} unique")

In [ ]:
from data import CharTokenizer, get_batch
from model import TinyMoreh

tokenizer = CharTokenizer(text)
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)

n = int(len(data) * 0.9)
train_data = data[:n].to('cuda')
val_data = data[n:].to('cuda')

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens: {len(val_data):,}")

In [ ]:
CONFIG = {
    "d_model": 256,
    "num_heads": 8,
    "num_layers": 6,
    "block_size": 256,
    "dropout": 0.1,
    "batch_size": 64,
    "learning_rate": 3e-4,
    "max_steps": 10000,
    "eval_interval": 500,
    "eval_steps": 20,
    "generate_every": 2000,
    "generate_tokens": 300,
}

model = TinyMoreh(
    vocab_size=tokenizer.vocab_size,
    d_model=CONFIG["d_model"],
    num_heads=CONFIG["num_heads"],
    num_layers=CONFIG["num_layers"],
    block_size=CONFIG["block_size"],
    dropout=CONFIG["dropout"],
).to('cuda')

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def estimate_loss():
    model.eval()
    losses = {}
    for name, d in [("train", train_data), ("val", val_data)]:
        total = 0.0
        for _ in range(CONFIG["eval_steps"]):
            x, y = get_batch(d, CONFIG["block_size"], CONFIG["batch_size"], 'cuda')
            _, loss = model(x, y)
            total += loss.item()
        losses[name] = total / CONFIG["eval_steps"]
    model.train()
    return losses


def generate_sample(num_tokens=300):
    model.eval()
    start = torch.zeros((1, 1), dtype=torch.long, device='cuda')
    tokens = model.generate(start, max_new_tokens=num_tokens, temperature=0.8)
    model.train()
    return tokenizer.decode(tokens[0].tolist())


optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])

print(f"Training for {CONFIG['max_steps']} steps...")
print("-" * 60)
start_time = time.time()

for step in range(CONFIG["max_steps"]):
    if step % CONFIG["eval_interval"] == 0 or step == CONFIG["max_steps"] - 1:
        losses = estimate_loss()
        elapsed = time.time() - start_time
        print(f"Step {step:>5d} | train: {losses['train']:.4f} | val: {losses['val']:.4f} | {elapsed:.0f}s")

    if step > 0 and step % CONFIG["generate_every"] == 0:
        sample = generate_sample()
        print(f"\n--- Sample at step {step} ---")
        print(sample)
        print("--- End sample ---\n")

    x, y = get_batch(train_data, CONFIG["block_size"], CONFIG["batch_size"], 'cuda')
    logits, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"\nDone in {time.time() - start_time:.0f}s")

In [ ]:
prompts = [
    "The nature of God",
    "The prophet spoke",
    "It is known that",
    "The law commands",
]

model.eval()
for prompt in prompts:
    tokens = [tokenizer.stoi[ch] for ch in prompt if ch in tokenizer.stoi]
    idx = torch.tensor([tokens], dtype=torch.long, device='cuda')
    output = model.generate(idx, max_new_tokens=300, temperature=0.8)
    text = tokenizer.decode(output[0].tolist())
    print(f"Prompt: \"{prompt}\"")
    print("-" * 40)
    print(text)
    print()

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
save_path = "checkpoints/tiny_moreh_gpu.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
    "vocab_size": tokenizer.vocab_size,
    "stoi": tokenizer.stoi,
    "itos": tokenizer.itos,
}, save_path)

print(f"Saved to {save_path} ({os.path.getsize(save_path) / 1024 / 1024:.1f} MB)")

from google.colab import files
files.download(save_path)